In [ ]:
import pandas as pd
import duckdb

In [ ]:
!ls /kaggle/input

Cheking the input, whrther the data is imported or not

In [ ]:
!ls /kaggle/input/steam-dataset-2025-multi-modal-gaming-analytics

Listing the directories

In [ ]:
BASE_PATH = (
    "/kaggle/input/"
    "steam-dataset-2025-multi-modal-gaming-analytics/"
    "steam_dataset_2025_csv_package_v1/"
    "steam_dataset_2025_csv"
)

import os
os.listdir(BASE_PATH)

In [ ]:
import pandas as pd

applications = pd.read_csv(f"{BASE_PATH}/applications.csv")
applications.shape

In [ ]:
applications.head()

In [ ]:
applications.columns

In [ ]:
con = duckdb.connect()

In [ ]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_genres.csv')
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/genres.csv')
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
LIMIT 5
""").df()

the above code need to be removed

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_genres AS
SELECT
    ag.appid,
    string_agg(g.name, ', ') AS genres
FROM read_csv_auto('{BASE_PATH}/application_genres.csv') ag
JOIN read_csv_auto('{BASE_PATH}/genres.csv') g
    ON ag.genre_id = g.id
GROUP BY ag.appid
""")

In [ ]:
con.execute("""
SELECT *
FROM app_genres
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/application_categories.csv')
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
SELECT *
FROM read_csv_auto('{BASE_PATH}/categories.csv')
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_categories AS
SELECT
    ac.appid,
    string_agg(c.name, ', ') AS categories
FROM read_csv_auto('{BASE_PATH}/application_categories.csv') ac
JOIN read_csv_auto('{BASE_PATH}/categories.csv') c
    ON ac.category_id = c.id
GROUP BY ac.appid
""")

In [ ]:
con.execute("""
SELECT *
FROM app_categories
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_developers AS
SELECT
    ad.appid,
    string_agg(DISTINCT d.name, ', ') AS developers
FROM read_csv_auto('{BASE_PATH}/application_developers.csv') ad
JOIN read_csv_auto('{BASE_PATH}/developers.csv') d
    ON ad.developer_id = d.id
GROUP BY ad.appid
""")

In [ ]:
con.execute("""
SELECT *
FROM app_developers
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW app_publishers AS
SELECT
    ap.appid,
    string_agg(DISTINCT p.name, ', ') AS publishers
FROM read_csv_auto('{BASE_PATH}/application_publishers.csv') ap
JOIN read_csv_auto('{BASE_PATH}/publishers.csv') p
    ON ap.publisher_id = p.id
GROUP BY ap.appid
""")

In [ ]:
con.execute("""
SELECT *
FROM app_publishers
LIMIT 5
""").df()

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE applications_wide AS
SELECT
    a.*,
    g.genres,
    c.categories,
    d.developers,
    p.publishers
FROM read_csv_auto(
        '{BASE_PATH}/applications.csv',
        ignore_errors=true
     ) a
LEFT JOIN app_genres g      ON a.appid = g.appid
LEFT JOIN app_categories c  ON a.appid = c.appid
LEFT JOIN app_developers d  ON a.appid = d.appid
LEFT JOIN app_publishers p  ON a.appid = p.appid
""")

In [ ]:
con.execute("""
SELECT COUNT(*) FROM applications_wide
""").df()

In [ ]:
con.execute("""
SELECT * FROM applications_wide LIMIT 5
""").df()

In [ ]:
df = con.execute("""
SELECT *
FROM applications_wide
""").df()

df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T.head(34)

In [ ]:
df.to_csv("/kaggle/working/applications_wide.csv", index=False)

In [ ]:
con.execute(f"""
SELECT DISTINCT recommendationid
FROM read_csv_auto('{BASE_PATH}/reviews.csv')
LIMIT 10
""").df()

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW reviews_aggregated AS
SELECT
    appid,
    COUNT(*) AS n_reviews,

    -- sentiment
    SUM(CASE WHEN voted_up THEN 1 ELSE 0 END) AS positive_reviews,
    SUM(CASE WHEN NOT voted_up THEN 1 ELSE 0 END) AS negative_reviews,

    -- engagement (correct column)
    AVG(author_playtime_forever) AS avg_playtime,

    -- review usefulness
    AVG(votes_up) AS avg_helpful_votes

FROM read_csv_auto(
    '{BASE_PATH}/reviews.csv',
    ignore_errors=true
)
GROUP BY appid
""")


In [ ]:
con.execute("""
SELECT *
FROM reviews_aggregated
LIMIT 10
""").df()

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE applications_wide_with_reviews AS
SELECT
    a.*,

    -- review volume
    r.n_reviews,
    r.positive_reviews,
    r.negative_reviews,

    -- derived metric
    r.positive_reviews * 1.0 / NULLIF(r.n_reviews, 0) AS positive_ratio,

    -- engagement
    r.avg_playtime,
    r.avg_helpful_votes

FROM applications_wide a
LEFT JOIN reviews_aggregated r
    ON a.appid = r.appid
""")

In [ ]:
con.execute("""
SELECT *
FROM applications_wide_with_reviews
LIMIT 3
""").df()

In [ ]:
df = con.execute("""
SELECT *
FROM applications_wide_with_reviews
""").df()

df.shape

In [ ]:
df.info()

In [ ]:
df.to_csv("/kaggle/working/applications_wide_with_reviews.csv", index=False)

In [ ]:
df.columns.tolist()

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.head(40)

In [ ]:
eda_cols = [
    # identifiers
    'appid', 'name',

    # metadata
    'type', 'is_free', 'required_age', 'release_date',

    # pricing
    'mat_initial_price', 'mat_final_price', 'mat_discount_percent',

    # content
    'genres', 'categories', 'developers', 'publishers',

    # sentiment
    'n_reviews', 'positive_reviews', 'negative_reviews', 'positive_ratio',

    # engagement
    'avg_playtime', 'avg_helpful_votes',

    # platforms
    'mat_supports_windows', 'mat_supports_linux', 'mat_supports_mac'
]

df_eda = df[eda_cols].copy()
df_eda.shape

In [ ]:
df_eda.to_csv("/kaggle/working/Steam_data_after_EDA.csv", index=False)

In [ ]:
df_eda.head(3)

In [ ]:
df_eda.info()